# COVID-19 Radiography DCGAN + Classifier Runner

Run each cell in order in Google Colab with a GPU runtime. This notebook trains the COVID-19 radiography version of the project on Colab, then saves the results back to Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

# If automatic search picks the wrong zip, paste the exact zip path here.
PROJECT_ZIP = "/content/drive/MyDrive/medical_gan_project_colab_ready.zip"
SEARCH_ROOT = Path('/content/drive/MyDrive')
WORK_ROOT = Path('/content/medical_gan_project_work')
REQUIRED_FILES = {'data.py', 'gan.py', 'train_gan.py', 'train_classifier.py'}

def zip_has_project(zip_path):
    try:
        with zipfile.ZipFile(zip_path, 'r') as archive:
            names = [Path(name) for name in archive.namelist() if not name.endswith('/')]
        by_parent = {}
        for name in names:
            by_parent.setdefault(str(name.parent), set()).add(name.name)
        for parent, files in by_parent.items():
            if REQUIRED_FILES.issubset(files):
                return True, parent
    except zipfile.BadZipFile:
        return False, None
    return False, None

if PROJECT_ZIP is None:
    zips = sorted(SEARCH_ROOT.rglob('*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not zips:
        raise FileNotFoundError('No zip files found in /content/drive/MyDrive')
    print('Checking zip files:')
    chosen = None
    for idx, zip_path in enumerate(zips[:50]):
        ok, parent = zip_has_project(zip_path)
        mark = 'PROJECT' if ok else 'skip'
        print(f'{idx}: {mark}: {zip_path}')
        if ok and chosen is None:
            chosen = zip_path
    if chosen is None:
        print('\nNo zip contained the required project files.')
        print('Required files:', sorted(REQUIRED_FILES))
        print('Upload a fresh zip of the folder that contains data.py, gan.py, train_gan.py, and train_classifier.py.')
        raise FileNotFoundError('No valid project zip found')
    PROJECT_ZIP = str(chosen)

print('\nUsing project zip:', PROJECT_ZIP)
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PROJECT_ZIP, 'r') as archive:
    archive.extractall(WORK_ROOT)

candidates = []
search_dirs = [WORK_ROOT] + [path for path in WORK_ROOT.rglob('*') if path.is_dir()]
for path in search_dirs:
    files = {child.name for child in path.iterdir() if child.is_file()}
    if REQUIRED_FILES.issubset(files):
        candidates.append(path)

if not candidates:
    print('Extracted top-level contents:')
    for child in sorted(WORK_ROOT.iterdir()):
        print(' -', child)
    raise FileNotFoundError('The zip extracted, but the project root was not found')

PROJECT_ROOT = candidates[0]
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
!pwd
!ls


In [ ]:
!python -m pip install --upgrade pip -q
!python -m pip install -q pandas numpy matplotlib scikit-learn flask kaggle pillow torch torchvision


In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Use Runtime > Change runtime type > T4 GPU, then rerun.')


In [ ]:
import getpass, json, os
from pathlib import Path

KAGGLE_USERNAME = input('Paste your Kaggle username: ').strip()
KAGGLE_KEY = getpass.getpass('Paste your Kaggle API key here: ')
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
with (kaggle_dir / 'kaggle.json').open('w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
print('Kaggle credentials saved.')


In [ ]:
!kaggle datasets list -s "covid19 radiography database"


In [ ]:
!python download_dataset.py --output_dir data/covid_radiography_raw


In [ ]:
# Fast Colab setting. Increase or set to None for the full dataset.
MAX_IMAGES_PER_CLASS = 1000

limit_arg = '' if MAX_IMAGES_PER_CLASS is None else f'--max_images_per_class {MAX_IMAGES_PER_CLASS}'
!python prepare_dataset.py --raw_dir data/covid_radiography_raw --output_dir data/covid_radiography --test_size 0.2 --overwrite {limit_arg}
!find data/covid_radiography -maxdepth 2 -type d | sort | head -30


In [ ]:
# Fast test: 5-10 epochs. Better quality: 50+ epochs.
GAN_EPOCHS = 10

!python train_gan.py --train_dir data/covid_radiography/train --out_name covid_dcgan.pth --epochs {GAN_EPOCHS} --batch_size 32 --image_size 64 --num_workers 2


In [ ]:
from pathlib import Path

SYNTHETIC_PER_CLASS = 500

if not Path('checkpoints/covid_dcgan.pth').exists() and Path('checkpoints/generator.pth').exists():
    !cp checkpoints/generator.pth checkpoints/covid_dcgan.pth

!ls -lah checkpoints/covid_dcgan.pth

!python generate_synthetic.py --checkpoint checkpoints/covid_dcgan.pth --match_real_dir data/covid_radiography/train --out_dir synthetic_covid_images --samples_per_class {SYNTHETIC_PER_CLASS} --batch_size 32


In [ ]:
# Fast test: 5 epochs. Better classifier: 20+ epochs.
CLASSIFIER_EPOCHS = 5

!python train_classifier.py --real_train_dir data/covid_radiography/train --synthetic_train_dir synthetic_covid_images --test_dir data/covid_radiography/test --out_name covid_classifier.pth --epochs {CLASSIFIER_EPOCHS} --batch_size 32 --num_workers 2


In [ ]:
from pathlib import Path
import json

required = ['checkpoints/covid_dcgan.pth', 'checkpoints/covid_classifier.pth', 'frontend/app.py']
for path in required:
    print(path, 'OK' if Path(path).exists() else 'MISSING')

missing = [path for path in required if not Path(path).exists()]
if missing:
    raise FileNotFoundError('Missing outputs: ' + ', '.join(missing))

metrics_path = Path('checkpoints/covid_classifier_metrics.json')
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    report = metrics.get('classification_report', {})
    print('Accuracy:', metrics.get('accuracy'))
    print('Macro F1:', report.get('macro avg', {}).get('f1-score'))
    print('Weighted F1:', report.get('weighted avg', {}).get('f1-score'))


In [ ]:
!mkdir -p /content/drive/MyDrive/medical_gan_covid_results
!cp -r checkpoints generated_samples synthetic_covid_images report.md /content/drive/MyDrive/medical_gan_covid_results/
!ls /content/drive/MyDrive/medical_gan_covid_results
